# Distance Recalculation

In [ ]:
import pandas as pd
import numpy as np

# Formule mathématique pour calculer la distance entre 2 points GPS en kilomètres
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0 # Rayon de la Terre en km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    # On multiplie par 1.15 pour compenser le fait que les rails ne sont 
    # jamais en ligne droite parfaite (facteur de courbure urbain standard)
    return R * c * 1.15

# 1. Charger votre fichier
file_path = 'delhi-metro-stations.csv'  # Assurez-vous que le chemin est correct
df = pd.read_csv(file_path)

# On s'assure que la colonne Branch existe (au cas où)
if 'Branch' not in df.columns:
    df['Branch'] = 'Main'

# Création d'une nouvelle colonne temporaire pour les calculs
df['Nouvelle_Distance'] = 0.0

# 2. Grouper par Ligne ET par Branche (sort=False permet de garder l'ordre actuel de votre CSV)
for (line, branch), group in df.groupby(['Line', 'Branch'], sort=False):
    distances_cumulees = [0.0] # La première station de cette ligne/branche est toujours à 0 km
    
    # On boucle uniquement sur les stations de CE groupe spécifique
    for i in range(1, len(group)):
        # Station précédente
        lat1 = group.iloc[i-1]['Latitude']
        lon1 = group.iloc[i-1]['Longitude']
        # Station actuelle
        lat2 = group.iloc[i]['Latitude']
        lon2 = group.iloc[i]['Longitude']
        
        # Distance entre ces deux stations
        dist_segment = haversine(lat1, lon1, lat2, lon2)
        
        # Ajout au cumul total
        cumul = distances_cumulees[-1] + dist_segment
        distances_cumulees.append(round(cumul, 2))
        
    # On injecte les distances calculées à leur bonne place dans le grand DataFrame
    df.loc[group.index, 'Nouvelle_Distance'] = distances_cumulees

# 3. Mettre à jour la colonne finale et nettoyer
df['Distance'] = df['Nouvelle_Distance']
df = df.drop(columns=['Nouvelle_Distance'])

# Afficher un petit aperçu pour vérifier
print(df[['Station Name', 'Line', 'Branch', 'Distance']].head(20))

# 4. Sauvegarder le fichier final
df.to_csv(file_path, index=False)
print("🎉 Terminé ! Les distances ont été recalculées et sauvegardées pour TOUTES les lignes et branches.")

        Station Name  Line           Branch  Distance
0           Vaishali  Blue  Vaishali Branch      0.00
1          Kaushambi  Blue  Vaishali Branch      1.82
2        Anand Vihar  Blue  Vaishali Branch      2.78
3         Karkarduma  Blue  Vaishali Branch      3.96
4        Preet Vihar  Blue  Vaishali Branch      5.44
5       Nirman Vihar  Blue  Vaishali Branch      6.57
6        Laxmi Nagar  Blue  Vaishali Branch      7.85
7        Yamuna Bank  Blue  Vaishali Branch      9.28
8   Dwarka Sector 21  Blue             Main      0.00
9    Dwarka Sector 8  Blue             Main      1.97
10   Dwarka Sector 9  Blue             Main      3.10
11  Dwarka Sector 10  Blue             Main      4.33
12  Dwarka Sector 11  Blue             Main      5.47
13  Dwarka Sector 12  Blue             Main      6.70
14  Dwarka Sector 13  Blue             Main      7.73
15  Dwarka Sector 14  Blue             Main      8.79
16            Dwarka  Blue             Main     10.45
17        Dwarka Mor  Blue  

# Wikipedia Coordinates Fetched

In [ ]:
import pandas as pd
import requests
import time
from urllib.parse import unquote, quote

# 1. Charger le fichier
file_path = 'delhi-metro-stations.csv'
df = pd.read_csv(file_path)

# Si vous voulez calculer pour une ligne spécifique qui pose problème (ex: la Blue Line)
#ligne_a_calculer = 'Grey'
#branche_a_calculer = 'Main' # ou 'Vaishali_Branch'

# Filtrer les stations de cette branche
#mask = (df['Line'] == ligne_a_calculer) & (df['Branch'] == branche_a_calculer)
#df = df[mask].copy()

def fetch_decimal_coords(wiki_link):
    # Ignorer si le lien est vide ou invalide
    if not isinstance(wiki_link, str) or 'wikipedia.org' not in wiki_link:
        return None, None
    
    clean_link = wiki_link.strip()
    # Extraire le titre de la page depuis l'URL (ex: "Dwarka_metro_station")
    raw_title = clean_link.split('/')[-1]
    title = quote(unquote(raw_title))
    
    # L'API magique de Wikipédia qui renvoie les coordonnées en format décimal
    url = f"https://en.wikipedia.org/w/api.php?action=query&prop=coordinates&titles={title}&format=json&redirects=1"
    
    try:
        # Wikipedia demande toujours un "User-Agent" pour savoir qui utilise son API
        headers = {'User-Agent': 'MetroMapperProject/1.0'}
        response = requests.get(url, headers=headers)
        
        if response.status_code == 429:
            print("🛑 Wikipédia nous demande de ralentir (Rate Limit) ! Pause de 5 secondes...")
            time.sleep(5)
            return fetch_decimal_coords(wiki_link) # On réessaie
            
        if response.status_code != 200:
            return None, None
        
        data = response.json()
        
        # Parcourir la réponse JSON pour trouver les coordonnées
        pages = data.get('query', {}).get('pages', {})
        for page_id, page_info in pages.items():
            if 'coordinates' in page_info:
                lat = page_info['coordinates'][0]['lat']
                lon = page_info['coordinates'][0]['lon']
                return lat, lon
    except Exception as e:
        print(f"Erreur lors de la récupération des coordonnées pour {wiki_link} : {e}")
        pass
        
    return None, None

print("Début du téléchargement des coordonnées (Cela peut prendre 1 à 2 minutes)...")

for index, row in df.iterrows():
    wiki_link = row['WikiLink']
    lat, lon = fetch_decimal_coords(wiki_link)
    
    if lat is not None and lon is not None:
        df.at[index, 'Latitude'] = round(lat, 6)
        df.at[index, 'Longitude'] = round(lon, 6)
        print(f"✅ {row['Station Name']} mise à jour : {lat}, {lon}")
    else:
        print(f"⚠️ Impossible de trouver les coordonnées pour {row['Station Name']} (On garde les anciennes)")
        
    # Petite pause de 0.1 seconde pour ne pas surcharger les serveurs de Wikipédia
    time.sleep(0.5)

# 3. Sauvegarder le fichier CSV avec les nouvelles coordonnées
df.to_csv(file_path, index=False)
print("🎉 Terminé ! Votre fichier a maintenant les coordonnées GPS décimales exactes de Wikipédia.")

Début du téléchargement des coordonnées (Cela peut prendre 1 à 2 minutes)...
✅ Shahdara mise à jour : 28.6735, 77.2899
✅ Welcome mise à jour : 28.6721, 77.2779
✅ Seelampur mise à jour : 28.6698, 77.2667
✅ Shastri Park mise à jour : 28.6682, 77.2501
✅ Kashmere Gate mise à jour : 28.6675, 77.228
✅ Tis Hazari mise à jour : 28.6672, 77.2165
✅ Pulbangash mise à jour : 28.6663, 77.207
✅ Pratap Nagar mise à jour : 28.6667, 77.1988
✅ Shastri Nagar mise à jour : 28.6701, 77.1818
✅ Inderlok mise à jour : 28.6734, 77.1703
🛑 Wikipédia nous demande de ralentir (Rate Limit) ! Pause de 5 secondes...
🛑 Wikipédia nous demande de ralentir (Rate Limit) ! Pause de 5 secondes...
🛑 Wikipédia nous demande de ralentir (Rate Limit) ! Pause de 5 secondes...
🛑 Wikipédia nous demande de ralentir (Rate Limit) ! Pause de 5 secondes...
🛑 Wikipédia nous demande de ralentir (Rate Limit) ! Pause de 5 secondes...
🛑 Wikipédia nous demande de ralentir (Rate Limit) ! Pause de 5 secondes...
🛑 Wikipédia nous demande de ralen